In [1]:
import gradio as gr
import pandas as pd
import pickle

In [8]:
df = pd.read_csv("../data/audio_mnist_metadata.csv")
print(df.shape)
print(df.head())

(30000, 7)
   digit  speaker_id  gender  accent   age                   origin  split
0      7          59  female  German  31.0  Europe, Germany, Berlin  train
1      7          59  female  German  31.0  Europe, Germany, Berlin  train
2      2          59  female  German  31.0  Europe, Germany, Berlin  train
3      3          59  female  German  31.0  Europe, Germany, Berlin  train
4      9          59  female  German  31.0  Europe, Germany, Berlin  train


In [21]:
# Get unique values for each column for the dropdown filters
def get_filter_options():
    filters = {}
    for col in df.columns:
        # Get unique values, excluding NaN
        unique_vals = df[col].dropna().astype(str).unique().tolist()
        # Sort and add "All" option
        filters[col] = ["All"] + sorted(unique_vals)
    return filters
 
filter_options = get_filter_options()
 
def filter_data(digit, speaker_id, gender, accent, age, origin, split):
    """Filter the dataframe based on selected criteria"""
    filtered_df = df.copy()
    
    # Apply filters (only if not "All" is selected)
    if digit != "All":
        filtered_df = filtered_df[filtered_df["digit"].astype(str) == digit]
    if speaker_id != "All":
        filtered_df = filtered_df[filtered_df["speaker_id"].astype(str) == speaker_id]
    if gender != "All":
        filtered_df = filtered_df[filtered_df["gender"].astype(str) == gender]
    if accent != "All":
        filtered_df = filtered_df[filtered_df["accent"].astype(str) == accent]
    if age != "All":
        filtered_df = filtered_df[filtered_df["age"].astype(str) == age]
    if origin != "All":
        filtered_df = filtered_df[filtered_df["origin"].astype(str) == origin]
    if split != "All":
        filtered_df = filtered_df[filtered_df["split"].astype(str) == split]
    
    # Return both the table and a count
    count = len(filtered_df)
    return filtered_df, f"Showing {count} entries"

def reset_filters():
    """Reset all filters to 'All'"""
    return "All", "All", "All", "All", "All", "All", "All", df, f"Showing {len(df)} entries"

### Remember to close the port with .close() or by restarting the kernel. If you don't, it sometimes struggles to re-run the code cause the port is busy.

In [ ]:
def create_app():
    with gr.Blocks() as interface:
        gr.Markdown(r"""
            # Spoken Digit Classification
            ### Project description
            This project uses the [Audio MNIST Dataset](https://huggingface.co/datasets/gilkeyio/AudioMNIST) to train a model that predicts numbers from 0 to 9 based on audio input. The neural network is a multilayer perceptron with an input of 26 neurons, two hidden layers of 64 and 32 neurons, and a final output layer of 10 neurons. It uses the Adam optimizer and the cross-entropy loss function.
        """)

        with gr.Tabs():
            # ========================================= DATA EXPLORATION TAB =========================================
            with gr.Tab("Data"):
                with gr.Group():
                    gr.Markdown("#### Dataset Exploration")
                    gr.Markdown("Select filters below to view specific rows from your dataset")
    
                    with gr.Accordion("Filters", open=False):
                        with gr.Row():
                            digit_filter = gr.Dropdown(
                                choices=filter_options["digit"],
                                value="All",
                                label="Digit"
                            )
                            speaker_id_filter = gr.Dropdown(
                                choices=filter_options["speaker_id"],
                                value="All",
                                label="Speaker ID"
                            )
                            gender_filter = gr.Dropdown(
                                choices=filter_options["gender"],
                                value="All",
                                label="Gender"
                            )
                        
                        with gr.Row():
                            accent_filter = gr.Dropdown(
                                choices=filter_options["accent"],
                                value="All",
                                label="Accent"
                            )
                            age_filter = gr.Dropdown(
                                choices=filter_options["age"],
                                value="All",
                                label="Age"
                            )
                            origin_filter = gr.Dropdown(
                                choices=filter_options["origin"],
                                value="All",
                                label="Origin"
                            )
                        
                        with gr.Row():
                            split_filter = gr.Dropdown(
                                choices=filter_options["split"],
                                value="All",
                                label="Split"
                            )

                        # Reset button
                        with gr.Row():
                            reset_btn = gr.Button("🔄 Reset Filters", variant="secondary")
                    
                    # Data table output
                    data_table = gr.Dataframe(
                        value=df,
                        interactive=False,
                        label="Data"
                    )
                    
                    # Result count
                    result_text = gr.Textbox(
                        value=f"Showing {len(df)} entries",
                        interactive=False,
                        label="Result Count"
                    )
                    
                    # Create event listeners for each filter
                    filters = [digit_filter, speaker_id_filter, gender_filter, accent_filter, age_filter, origin_filter, split_filter]
                    
                    for filter_elem in filters:
                        filter_elem.change(
                            fn=filter_data,
                            inputs=filters,
                            outputs=[data_table, result_text]
                        )

            # ======================================== TRAINING INTERFACE TAB ========================================
            with gr.Tab("Train"):
                gr.Markdown("#### Training Interface")

            # ========================================= MODEL EVALUATION TAB =========================================
            with gr.Tab("Model"):
                gr.Markdown("#### Model Evaluation")

        gr.Markdown(r"""
            *Created by: Amets Martiarena, Teva Philippe, Alvaro Crespo | Check the original repo [here](https://github.com/AmetsMarti/SDOML-project1)*.
        """)
    return interface

In [20]:
try:
    interface.close()
except:
    pass

# Set to true to publish the app
send_it = False

interface = create_app()
interface.launch(server_port=7860, share=send_it)

Closing server running on port: 7860
* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
